# Taxi Anomaly Detection with OpenShift AI

This notebook loads taxi trip data from MinIO and detects anomalies using scikit-learn's Isolation Forest.

The quickstart stores data in MinIO (`data` bucket). In production, Zetaris can serve as the upstream data platform; MinIO holds the working dataset for experimentation on OpenShift AI.

In [ ]:
import io
import os

import boto3
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import IsolationForest

bucket = os.getenv("TAXI_DATA_BUCKET", os.getenv("AWS_S3_BUCKET", "data"))
object_key = os.getenv("TAXI_DATA_OBJECT", "taxi_trips.csv")

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

body = s3.get_object(Bucket=bucket, Key=object_key)["Body"].read()
df = pd.read_csv(io.BytesIO(body))
print(f"Loaded {len(df)} trips from s3://{bucket}/{object_key}")
df.head()

In [ ]:
features = ["passenger_count", "trip_distance", "fare_amount", "tip_amount"]
X = df[features]

model = IsolationForest(contamination=0.02, random_state=42)
df["anomaly_score"] = model.fit_predict(X)
df["is_anomaly"] = df["anomaly_score"] == -1

anomalies = df[df["is_anomaly"]]
print(f"Detected {len(anomalies)} anomalous trips ({len(anomalies) / len(df):.1%})")
anomalies.sort_values("fare_amount", ascending=False).head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
normal = df[~df["is_anomaly"]]
anomaly = df[df["is_anomaly"]]

ax.scatter(normal["trip_distance"], normal["fare_amount"], alpha=0.5, label="Normal")
ax.scatter(anomaly["trip_distance"], anomaly["fare_amount"], color="red", label="Anomaly")
ax.set_xlabel("Trip distance (miles)")
ax.set_ylabel("Fare amount ($)")
ax.set_title("Taxi fare anomalies")
ax.legend()
plt.show()